<a href="https://colab.research.google.com/github/KaveeshwaraBandara/EN4554-A2-CNNs-CIFAR10/blob/main/220059J_a02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch, torchvision, numpy as np, random, json, os, time
import torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Raw data -> float tensors in [0,1], shape (N,3,32,32)
tr_raw = torchvision.datasets.CIFAR10('./data', train=True,  download=True)
te_raw = torchvision.datasets.CIFAR10('./data', train=False, download=True)
X = torch.tensor(tr_raw.data).permute(0,3,1,2).float() / 255.
Y = torch.tensor(tr_raw.targets)

# Fixed 45k / 5k split
g = torch.Generator().manual_seed(SEED)
perm = torch.randperm(50000, generator=g)
X_tr, Y_tr = X[perm[:45000]], Y[perm[:45000]]
X_va, Y_va = X[perm[45000:]], Y[perm[45000:]]

# Per-channel stats from the 45k TRAINING images only
mean = X_tr.mean(dim=(0,2,3), keepdim=True)
std  = X_tr.std(dim=(0,2,3), keepdim=True)
print('mean', mean.flatten(), 'std', std.flatten())

norm = lambda x: (x - mean) / std
X_tr, X_va = norm(X_tr).to(device), norm(X_va).to(device)   # whole dataset lives on the GPU
Y_tr, Y_va = Y_tr.to(device), Y_va.to(device)

# Test set: normalized with TRAIN stats, kept on CPU, untouched until Task 4
X_te = norm(torch.tensor(te_raw.data).permute(0,3,1,2).float() / 255.)
Y_te = torch.tensor(te_raw.targets)

def batches(X, Y, bs, shuffle=True):
    idx = torch.randperm(len(X)).to(X.device) if shuffle else torch.arange(len(X), device=X.device)
    for i in range(0, len(X), bs):
        j = idx[i:i+bs]
        yield X[j], Y[j]

100%|██████████| 170M/170M [24:35<00:00, 116kB/s]


mean tensor([0.4917, 0.4823, 0.4467]) std tensor([0.2471, 0.2435, 0.2616])


In [2]:
from google.colab import drive
drive.mount('/content/drive')

RESULTS = '/content/drive/MyDrive/EN4554_A2/results'
os.makedirs(RESULTS, exist_ok=True)

def save_results(name, obj):
    with open(f'{RESULTS}/{name}.json', 'w') as f:
        json.dump(obj, f)

def load_results(name):
    p = f'{RESULTS}/{name}.json'
    return json.load(open(p)) if os.path.exists(p) else None

Mounted at /content/drive
